# SDK Wheel Smoke Test

Runs as the `sdk_smoke` job of the SDK bundle. The job environment installs the Genie Ontology Modeled Knowledge SDK wheel that `bundle deploy` built and uploaded to the `sdk_wheels` volume; this notebook imports the installed package, checks it is not a source checkout, prints `__version__` and exits with it.

**Widget:** `expected_version` (optional; when set, the run fails if `__version__` differs).

In [0]:
%pip install --upgrade databricks-sdk
# The SDK wheel itself comes from the job environment (resources/sdk_smoke.job.yml);
# restartPython() keeps environment libraries installed.
dbutils.library.restartPython()

In [0]:
import json
from importlib import metadata
from pathlib import Path

dbutils.widgets.text("expected_version", "")
expected = dbutils.widgets.get("expected_version").strip()

import genie_ontology_modeled_knowledge_sdk as sdk

pkg_file = Path(sdk.__file__).resolve()
dist_version = metadata.version(sdk.DIST_NAME)
print(f"genie_ontology_modeled_knowledge_sdk __version__ {sdk.__version__}")
print(f"distribution {sdk.DIST_NAME} {dist_version}")
print(f"imported from {pkg_file}")

problems = []
if "site-packages" not in pkg_file.parts and "dist-packages" not in pkg_file.parts:
    problems.append(f"package imported from {pkg_file}, not from an installed wheel")
if sdk.__version__ != dist_version:
    problems.append(f"__version__ {sdk.__version__} != wheel metadata {dist_version}")
if expected and sdk.__version__ != expected:
    problems.append(f"__version__ {sdk.__version__} != expected_version {expected}")
if problems:
    raise RuntimeError("SDK smoke failed: " + "; ".join(problems))

# Import every library module once, so a missing package file fails here.
for mod in ("config", "mv_yaml", "manifest", "geniespace", "pages", "benchmark", "scaffold", "targets"):
    __import__(f"genie_ontology_modeled_knowledge_sdk.{mod}")

dbutils.notebook.exit(json.dumps({"__version__": sdk.__version__, "file": str(pkg_file)}))